In [1]:
import os
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
warnings.filterwarnings('ignore')

In [2]:
#Datasets
master_merge = pd.read_csv(r"C:\Users\USER\PycharmProjects\ThesisProject\Data\master_merged.csv")#news+ect+price (2020-2023)
df_news = pd.read_csv(r"C:\Users\USER\PycharmProjects\ThesisProject\Data\News\news_price.csv")#news+price (2020-2023) 
df_ect = pd.read_csv(r"C:\Users\USER\PycharmProjects\ThesisProject\Data\Transcripts\ect_price.csv")#ect+price (2020-2023)
news_2019 = pd.read_csv(r"C:\Users\USER\PycharmProjects\ThesisProject\Data\News\news_2019.csv")#news+price 2019
ect_2024 = pd.read_csv(r"C:\Users\USER\PycharmProjects\ThesisProject\Data\Transcripts\ect_2024.csv")#ect+price 2024

#### Reviewing and Handling NANs

In [3]:
df_news.columns

Index(['date', 'ticker', 'Adj_Close', 'Log_Return', 'Market_Return',
       'excess_return', 'Excess_Return_Winsor', 'Target_T1', 'Target_T3',
       'Target_T5', 'movement_T1', 'movement_T3', 'movement_T5',
       'lag_return_1d_winsor', 'volatility_5d_winsor',
       'title_svm_net_sentiment_mean', 'title_svm_net_sentiment_max',
       'title_svm_net_sentiment_min', 'title_svm_net_sentiment_std',
       'title_lr_net_sentiment_mean', 'title_lr_net_sentiment_max',
       'title_lr_net_sentiment_min', 'title_lr_net_sentiment_std',
       'article_svm_net_sentiment_mean', 'article_svm_net_sentiment_max',
       'article_svm_net_sentiment_min', 'article_svm_net_sentiment_std',
       'article_lr_net_sentiment_mean', 'article_lr_net_sentiment_max',
       'article_lr_net_sentiment_min', 'article_lr_net_sentiment_std',
       'news_finbert_mean', 'news_finbert_max', 'news_finbert_min',
       'news_finbert_std', 'news_gpt_mean', 'news_gpt_max', 'news_gpt_min',
       'news_gpt_std', 'Doc_C

In [4]:
#news+ect+price (2020-2023)
nan_distribution_master = master_merge.isna().sum()
print(nan_distribution_master[nan_distribution_master > 0])

title_svm_net_sentiment_mean           2101
title_svm_net_sentiment_max            2101
title_svm_net_sentiment_min            2101
title_svm_net_sentiment_std            2101
title_lr_net_sentiment_mean            2101
                                      ...  
txt_lr_net_sentiment_weighted_mean    14463
ect_finbert_range                     14463
ect_finbert_weighted_mean             14463
ect_gpt_range                         14463
ect_gpt_weighted_mean                 14463
Length: 70, dtype: int64


In [5]:
#news+price (2020-2023)
nan_distribution_news = df_news.isna().sum()
print(nan_distribution_news[nan_distribution_news > 0])

title_svm_net_sentiment_mean               2101
title_svm_net_sentiment_max                2101
title_svm_net_sentiment_min                2101
title_svm_net_sentiment_std                2101
title_lr_net_sentiment_mean                2101
title_lr_net_sentiment_max                 2101
title_lr_net_sentiment_min                 2101
title_lr_net_sentiment_std                 2101
article_svm_net_sentiment_mean             2101
article_svm_net_sentiment_max              2101
article_svm_net_sentiment_min              2101
article_svm_net_sentiment_std              2101
article_lr_net_sentiment_mean              2101
article_lr_net_sentiment_max               2101
article_lr_net_sentiment_min               2101
article_lr_net_sentiment_std               2101
news_finbert_mean                          2101
news_finbert_max                           2101
news_finbert_min                           2101
news_finbert_std                           2101
news_gpt_mean                           

In [6]:
#ect+price (2020-2023)
nan_distribution_ect = df_ect.isna().sum()
print(nan_distribution_ect[nan_distribution_ect > 0])

txt_svm_net_sentiment_mean             14463
txt_svm_net_sentiment_max              14463
txt_svm_net_sentiment_min              14463
txt_svm_net_sentiment_std              14463
txt_lr_net_sentiment_mean              14463
txt_lr_net_sentiment_max               14463
txt_lr_net_sentiment_min               14463
txt_lr_net_sentiment_std               14463
ect_finbert_mean                       14463
ect_finbert_max                        14463
ect_finbert_min                        14463
ect_finbert_std                        14463
ect_gpt_mean                           14463
ect_gpt_max                            14463
ect_gpt_min                            14463
ect_gpt_std                            14463
Total_Segment_Count                    14463
qa_txt_svm_net_sentiment_mean          14475
qa_txt_lr_net_sentiment_mean           14475
qa_finbert_net_score_mean              14475
qa_gpt_net_sentiment_mean              14475
prepared_txt_svm_net_sentiment_mean    14463
prepared_t

In [7]:
#news+price 2019
nan_distribution_2019 = news_2019.isna().sum()
print(nan_distribution_2019[nan_distribution_2019 > 0])

title_svm_net_sentiment_mean               603
title_svm_net_sentiment_max                603
title_svm_net_sentiment_min                603
title_svm_net_sentiment_std                603
title_lr_net_sentiment_mean                603
title_lr_net_sentiment_max                 603
title_lr_net_sentiment_min                 603
title_lr_net_sentiment_std                 603
article_svm_net_sentiment_mean             603
article_svm_net_sentiment_max              603
article_svm_net_sentiment_min              603
article_svm_net_sentiment_std              603
article_lr_net_sentiment_mean              603
article_lr_net_sentiment_max               603
article_lr_net_sentiment_min               603
article_lr_net_sentiment_std               603
news_finbert_mean                          603
news_finbert_max                           603
news_finbert_min                           603
news_finbert_std                           603
news_gpt_mean                              603
news_gpt_max 

In [8]:
#ect+price 2024
nan_distribution_2024 = ect_2024.isna().sum()
print(nan_distribution_2024[nan_distribution_master > 0])

txt_svm_net_sentiment_mean             3597
txt_svm_net_sentiment_max              3597
txt_svm_net_sentiment_min              3597
txt_svm_net_sentiment_std              3597
txt_lr_net_sentiment_mean              3597
txt_lr_net_sentiment_max               3597
txt_lr_net_sentiment_min               3597
txt_lr_net_sentiment_std               3597
ect_finbert_mean                       3597
ect_finbert_max                        3597
ect_finbert_min                        3597
ect_finbert_std                        3597
ect_gpt_mean                           3597
ect_gpt_max                            3597
ect_gpt_min                            3597
ect_gpt_std                            3597
Total_Segment_Count                    3597
qa_txt_svm_net_sentiment_mean          3605
qa_txt_lr_net_sentiment_mean           3605
qa_finbert_net_score_mean              3605
qa_gpt_net_sentiment_mean              3605
prepared_txt_svm_net_sentiment_mean    3597
prepared_txt_lr_net_sentiment_me

In [9]:
#Handling the NaNs
#Explicit News Feature List
news_cols = [
    'title_svm_net_sentiment_mean', 'title_svm_net_sentiment_max', 'title_svm_net_sentiment_min', 'title_svm_net_sentiment_std',
    'title_lr_net_sentiment_mean', 'title_lr_net_sentiment_max', 'title_lr_net_sentiment_min', 'title_lr_net_sentiment_std',
    'article_svm_net_sentiment_mean', 'article_svm_net_sentiment_max', 'article_svm_net_sentiment_min', 'article_svm_net_sentiment_std',
    'article_lr_net_sentiment_mean', 'article_lr_net_sentiment_max', 'article_lr_net_sentiment_min', 'article_lr_net_sentiment_std',
    'news_finbert_mean', 'news_finbert_max', 'news_finbert_min', 'news_finbert_std', 
    'news_gpt_mean', 'news_gpt_max', 'news_gpt_min', 'news_gpt_std', 
    'Doc_Count', 
    'title_svm_net_sentiment_range', 'title_svm_net_sentiment_weighted_mean', 
    'title_lr_net_sentiment_range', 'title_lr_net_sentiment_weighted_mean',
    'article_svm_net_sentiment_range', 'article_svm_net_sentiment_weighted_mean',
    'article_lr_net_sentiment_range', 'article_lr_net_sentiment_weighted_mean', 
    'news_finbert_range', 'news_finbert_weighted_mean', 
    'news_gpt_range', 'news_gpt_weighted_mean'
]

#Explicit Transcript Feature List
ect_cols = [
    'txt_svm_net_sentiment_mean', 'txt_svm_net_sentiment_max', 'txt_svm_net_sentiment_min', 'txt_svm_net_sentiment_std', 
    'txt_lr_net_sentiment_mean', 'txt_lr_net_sentiment_max', 'txt_lr_net_sentiment_min', 'txt_lr_net_sentiment_std', 
    'ect_finbert_mean', 'ect_finbert_max', 'ect_finbert_min', 'ect_finbert_std', 
    'ect_gpt_mean', 'ect_gpt_max', 'ect_gpt_min', 'ect_gpt_std', 
    'Total_Segment_Count',
    'qa_txt_svm_net_sentiment_mean', 'qa_txt_lr_net_sentiment_mean', 'qa_finbert_net_score_mean', 'qa_gpt_net_sentiment_mean', 
    'prepared_txt_svm_net_sentiment_mean', 'prepared_txt_lr_net_sentiment_mean', 'prepared_finbert_net_score_mean', 'prepared_gpt_net_sentiment_mean', 
    'txt_svm_net_sentiment_range', 'txt_svm_net_sentiment_weighted_mean', 
    'txt_lr_net_sentiment_range', 'txt_lr_net_sentiment_weighted_mean', 
    'ect_finbert_range', 'ect_finbert_weighted_mean', 
    'ect_gpt_range', 'ect_gpt_weighted_mean'
]

In [10]:
#News + Price Datasets (2019 Slice & 2020-2023 Main Panel) 
#Action: 1-Day Forward-Fill then remaining gaps filled with Neutral 0
for df_news_slice in [news_2019, df_news]:
    #Extracts only the subset of columns that actually exist in the current dataframe slice
    available_news = [c for c in news_cols if c in df_news_slice.columns]
    df_news_slice[available_news] = df_news_slice[available_news].ffill(limit=1).fillna(0)

#Transcripts + Price Dataset (2024 Historical Slice)
#Action: Isolates pure event disclosure rows
available_ect_2024 = [c for c in ect_cols if c in ect_2024.columns]
df_ect_2024_cleaned = ect_2024.dropna(subset=['ect_finbert_mean']).reset_index(drop=True)
df_ect_2024_cleaned = df_ect_2024_cleaned.drop_duplicates(subset=['ticker', 'ect_finbert_mean'],  keep='first').reset_index(drop=True)

#Transcripts + Price Dataset (2020-2023 Panel) 
#Action: Split into Event-Study (Universe A) vs. Continuous Market Matrix (Universe B)
available_ect_main = [c for c in ect_cols if c in df_ect.columns]

#Universe A: Pure Corporate Disclosure Days
df_ect_events_only = df_ect.dropna(subset=['ect_finbert_mean']).reset_index(drop=True)
df_ect_events_only = df_ect_events_only.drop_duplicates(subset=['ticker', 'ect_finbert_mean'],keep='first').reset_index(drop=True)

#Universe B: Continuous Daily Pipeline (No rows lost, empty quarters set to 0)
df_ect_continuous = df_ect.copy()
df_ect_continuous[available_ect_main] = df_ect_continuous[available_ect_main].fillna(0)

#Master Combined Dataset (News + Transcripts + Price 2020-2023)
df_modeling_master = master_merge.copy()

#News:1-Day Forward-Fill (Decay) then remaining gaps filled with Neutral 0
df_modeling_master[news_cols] = df_modeling_master[news_cols].ffill(limit=1).fillna(0)

#Apply neutral values to all the missing fields
df_modeling_master[ect_cols] = df_modeling_master[ect_cols].fillna(0)

#Drop only the trailing rows that do not have look-forward market targets available (max target T5)
df_modeling_master = df_modeling_master.dropna(subset=['movement_T5']).reset_index(drop=True)

In [11]:
#Verification for the Master Dataset
print(f"Master Matrix Shape:{df_modeling_master.shape}")
print(f"Master Remaining Nulls:{df_modeling_master.isna().sum().sum()}")

Master Matrix Shape:(15090, 85)
Master Remaining Nulls:0


In [12]:
#Verification for news and ECT datasets - Shape
print(f"News+Price(2020-2023): {df_news.shape}")
print(f"News+Price(2019): {news_2019.shape}")
print(f"ECT+Price(2020-2023): {df_ect_events_only.shape}") #Pure disclosure baseline rows
print(f"Continous ECT+Price(2020-2023): {df_ect_continuous.shape}")
print(f"ECT+Price(2024):{df_ect_2024_cleaned.shape}")

News+Price(2020-2023): (15090, 52)
News+Price(2019): (3642, 52)
ECT+Price(2020-2023): (228, 48)
Continous ECT+Price(2020-2023): (15090, 48)
ECT+Price(2024):(52, 48)


In [13]:
#Verification for news and ECT datasets - NaN Check
print(f"News+Price(2020-2023): {df_news.isna().sum().sum()}")
print(f"News+Price(2019): {news_2019.isna().sum().sum()}")
print(f"ECT+Price(2020-2023): {df_ect_events_only.isna().sum().sum()}")
print(f"Continous ECT+Price(2020-2023): {df_ect_continuous.isna().sum().sum()}")
print(f"ECT+Price(2024):{df_ect_2024_cleaned.isna().sum().sum()}")

News+Price(2020-2023): 0
News+Price(2019): 0
ECT+Price(2020-2023): 20
Continous ECT+Price(2020-2023): 0
ECT+Price(2024):16


In [14]:
#Checking the above NAN values
df_ect_events_only.isna().sum()

date                                   0
ticker                                 0
Adj_Close                              0
Log_Return                             0
Market_Return                          0
excess_return                          0
Excess_Return_Winsor                   0
Target_T1                              0
Target_T3                              0
Target_T5                              0
movement_T1                            0
movement_T3                            0
movement_T5                            0
lag_return_1d_winsor                   0
volatility_5d_winsor                   0
txt_svm_net_sentiment_mean             0
txt_svm_net_sentiment_max              0
txt_svm_net_sentiment_min              0
txt_svm_net_sentiment_std              0
txt_lr_net_sentiment_mean              0
txt_lr_net_sentiment_max               0
txt_lr_net_sentiment_min               0
txt_lr_net_sentiment_std               0
ect_finbert_mean                       0
ect_finbert_max 

In [15]:
df_ect_2024_cleaned.isna().sum()

date                                   0
ticker                                 0
Adj_Close                              0
Log_Return                             0
Market_Return                          0
excess_return                          0
Excess_Return_Winsor                   0
Target_T1                              0
Target_T3                              0
Target_T5                              0
movement_T1                            0
movement_T3                            0
movement_T5                            0
lag_return_1d_winsor                   0
volatility_5d_winsor                   0
txt_svm_net_sentiment_mean             0
txt_svm_net_sentiment_max              0
txt_svm_net_sentiment_min              0
txt_svm_net_sentiment_std              0
txt_lr_net_sentiment_mean              0
txt_lr_net_sentiment_max               0
txt_lr_net_sentiment_min               0
txt_lr_net_sentiment_std               0
ect_finbert_mean                       0
ect_finbert_max 

As per updated experiemnts there will be no comparison for prepared remarks vs qa thus the NANs identified above will not be a problem since these columns will be dropped. All the datasets are now suitable for modelling.

In [16]:
#Saving Data
output_path1 = r"C:\Users\USER\PycharmProjects\ThesisProject\Data\master_merged_overlap.csv"
output_path2 = r"C:\Users\USER\PycharmProjects\ThesisProject\Data\ect_overlap.csv"
output_path3 = r"C:\Users\USER\PycharmProjects\ThesisProject\Data\news_overlap.csv"

df_modeling_master.to_csv(output_path1, index=False)
df_ect_events_only.to_csv(output_path2, index=False)
df_news.to_csv(output_path3, index=False)
print('Success!')

Success!
